In [0]:
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
CREATE OR REPLACE TABLE silver.ref_companies AS
SELECT * FROM VALUES
  (320193,  'Apple'),
  (1018724, 'Amazon'),
  (789019,  'Microsoft')
AS t(cik, company);

In [0]:
CREATE OR REPLACE TABLE silver.ref_revenue_tags AS
SELECT * FROM VALUES
  ('Revenues'),
  ('RevenueFromContractWithCustomerExcludingAssessedTax'),
  ('RevenueFromContractWithCustomerIncludingAssessedTax'),
  ('SalesRevenueNet')
AS t(tag);

In [0]:
SELECT * FROM silver.ref_companies;
SELECT * FROM silver.ref_revenue_tags;
SELECT * FROM bronze.num LIMIT 20;

In [0]:
DESCRIBE TABLE bronze.num;        -- column names and types
SELECT COUNT(*) FROM bronze.num;  -- how many rows
SHOW TABLES IN silver;  

In [0]:
SELECT n.ddate, n.qtrs, n.tag, n.version, n.uom, n.dimh, n.iprx, n.coreg, n.value
FROM bronze.num n
JOIN bronze.sub s ON s.adsh = n.adsh AND s.load_period = n.load_period
WHERE n.load_period = '2024q1'
  AND CAST(s.cik AS BIGINT) = 1018724
  AND s.form = '10-K'
  AND n.tag = 'RevenueFromContractWithCustomerExcludingAssessedTax'
  AND n.dimn = '0'
ORDER BY n.ddate, n.iprx;

In [0]:
CREATE OR REPLACE TABLE silver.filings AS
SELECT
  s.adsh,
  c.company,
  CAST(s.cik AS BIGINT)               AS cik,
  s.name                              AS registrant_name,
  s.form,
  to_date(s.period, 'yyyyMMdd')       AS balance_date,
  to_date(s.filed,  'yyyyMMdd')       AS filed_date,
  TRY_CAST(s.fy AS INT)               AS fiscal_year,
  s.fp                                AS fiscal_period,
  s.load_period
FROM bronze.sub s
JOIN silver.ref_companies c
  ON CAST(s.cik AS BIGINT) = c.cik
WHERE s.form IN ('10-K', '10-Q');

SELECT * FROM silver.filings LIMIT 10;

SELECT COUNT(*) AS filings FROM silver.filings; 

In [0]:
SELECT n.adsh, n.tag, n.ddate, n.qtrs, n.dimh,
       COUNT(*) AS copies, COUNT(DISTINCT n.value) AS distinct_values
FROM bronze.num n
JOIN silver.filings f ON f.adsh = n.adsh AND f.load_period = n.load_period
JOIN silver.ref_revenue_tags t ON t.tag = n.tag
WHERE n.dimn IN ('0','1') AND n.uom = 'USD'
GROUP BY n.adsh, n.tag, n.version, n.ddate, n.qtrs, n.dimh, n.coreg
HAVING COUNT(DISTINCT n.value) > 1;

SELECT company, form, balance_date, filed_date, fiscal_year, fiscal_period, load_period
FROM silver.filings
ORDER BY company, balance_date;

In [0]:
CREATE OR REPLACE TABLE silver.facts_revenue AS
WITH base AS (
  SELECT
    f.company, f.cik, f.adsh, f.form, f.fiscal_year, f.fiscal_period, f.load_period,
    n.tag, n.version, n.uom, n.coreg, n.dimh,
    to_date(n.ddate, 'yyyyMMdd')      AS period_end,
    TRY_CAST(n.qtrs AS INT)           AS qtrs,
    TRY_CAST(n.iprx AS INT)           AS iprx,
    TRY_CAST(n.dimn AS INT)           AS dimn,
    TRY_CAST(n.value AS DECIMAL(24,4)) AS value,
    d.segments
  FROM bronze.num n
  JOIN silver.filings f
    ON f.adsh = n.adsh AND f.load_period = n.load_period
  JOIN silver.ref_revenue_tags t
    ON t.tag = n.tag
  LEFT JOIN bronze.dim d
    ON d.dimhash = n.dimh AND d.load_period = n.load_period
  WHERE n.uom = 'USD'
    AND n.dimn IN ('0', '1')
),
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY adsh, tag, version, period_end, qtrs, uom, dimh, coreg
           ORDER BY iprx
         ) AS rn
  FROM base
)
SELECT
  company, cik, adsh, form, fiscal_year, fiscal_period, load_period,
  tag, version, period_end, qtrs, uom, value, dimn,
  CASE WHEN dimn = 1 THEN split_part(segments, '=', 1) END                    AS axis,
  CASE WHEN dimn = 1 THEN replace(split_part(segments, '=', 2), ';', '') END  AS member
FROM ranked
WHERE rn = 1;

In [0]:
-- 1) No duplicates left: both numbers must be equal
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT concat_ws('|', adsh, tag, version, period_end, qtrs, dimn, axis, member)) AS distinct_facts
FROM silver.facts_revenue;

-- 2) Amazon annual totals: should be one row per year
SELECT company, period_end, qtrs, value
FROM silver.facts_revenue
WHERE company = 'Amazon' AND dimn = 0 AND qtrs = 4 AND form = '10-K'
ORDER BY period_end;

In [0]:
SELECT company, axis, member,
       COUNT(*)                   AS fact_rows,
       COUNT(DISTINCT period_end) AS periods
FROM silver.facts_revenue
WHERE dimn = 1
GROUP BY company, axis, member
ORDER BY company, axis, member;

In [0]:
CREATE OR REPLACE TABLE silver.ref_segment_map AS
SELECT * FROM VALUES
 -- company, axis, member, segment_name, segment_type, in_top_level, in_detail_level, include
 -- ===== AMAZON =====
 ('Amazon','BusinessSegments','AmazonWebServicesSegment','AWS','business_unit',true,true,true),
 ('Amazon','BusinessSegments','InternationalSegment','International','business_unit',true,true,true),
 ('Amazon','BusinessSegments','NorthAmericaSegment','North America','business_unit',true,true,true),
 ('Amazon','EquityMethodInvestmentNonconsolidatedInvestee','EquityMethodInvestmentNonconsolidatedInvesteeOrGroupOfInvestees','Equity-method investee (not Amazon revenue)','excluded',false,false,false),
 ('Amazon','Geographical','US','United States','geographic',true,true,true),
 ('Amazon','Geographical','DE','Germany','geographic',true,true,true),
 ('Amazon','Geographical','GB','United Kingdom','geographic',true,true,true),
 ('Amazon','Geographical','JP','Japan','geographic',true,true,true),
 ('Amazon','Geographical','NonUs','Rest of world','geographic',true,true,true),
 ('Amazon','ProductOrService','Product','Product (total)','product_service',true,false,true),
 ('Amazon','ProductOrService','Service','Service (total)','product_service',true,false,true),
 ('Amazon','ProductOrService','OnlineStores','Online stores','product_service',false,true,true),
 ('Amazon','ProductOrService','PhysicalStores','Physical stores','product_service',false,true,true),
 ('Amazon','ProductOrService','ThirdPartySellerServices','Third-party seller services','product_service',false,true,true),
 ('Amazon','ProductOrService','SubscriptionServices','Subscription services','product_service',false,true,true),
 ('Amazon','ProductOrService','Advertising','Advertising services','product_service',false,true,true),
 ('Amazon','ProductOrService','AdvertisingServices','Advertising services','product_service',false,true,true),
 ('Amazon','ProductOrService','AmazonWebServices','AWS','product_service',false,true,true),
 ('Amazon','ProductOrService','OtherServices','Other','product_service',false,true,true),
 -- ===== APPLE =====
 ('Apple','BusinessSegments','AmericasSegment','Americas','geographic',true,true,true),
 ('Apple','BusinessSegments','EuropeSegment','Europe','geographic',true,true,true),
 ('Apple','BusinessSegments','GreaterChinaSegment','Greater China','geographic',true,true,true),
 ('Apple','BusinessSegments','JapanSegment','Japan','geographic',true,true,true),
 ('Apple','BusinessSegments','RestOfAsiaPacificSegment','Rest of Asia Pacific','geographic',true,true,true),
 ('Apple','Geographical','US','United States','geographic',true,true,true),
 ('Apple','Geographical','CN','China','geographic',true,true,true),
 ('Apple','Geographical','OtherCountries','Other countries','geographic',true,true,true),
 ('Apple','ProductOrService','Product','Products (total)','product_service',true,false,true),
 ('Apple','ProductOrService','Service','Services','product_service',true,true,true),
 ('Apple','ProductOrService','IPhone','iPhone','product_service',false,true,true),
 ('Apple','ProductOrService','Mac','Mac','product_service',false,true,true),
 ('Apple','ProductOrService','IPad','iPad','product_service',false,true,true),
 ('Apple','ProductOrService','WearablesHomeandAccessories','Wearables, Home and Accessories','product_service',false,true,true),
 -- ===== MICROSOFT =====
 ('Microsoft','BusinessSegments','IntelligentCloud','Intelligent Cloud','business_unit',true,true,true),
 ('Microsoft','BusinessSegments','MorePersonalComputing','More Personal Computing','business_unit',true,true,true),
 ('Microsoft','BusinessSegments','ProductivityAndBusinessProcesses','Productivity and Business Processes','business_unit',true,true,true),
 ('Microsoft','Geographical','US','United States','geographic',true,true,true),
 ('Microsoft','Geographical','NonUs','Other countries','geographic',true,true,true),
 ('Microsoft','ProductOrService','Product','Product (total)','product_service',true,false,true),
 ('Microsoft','ProductOrService','ServiceOther','Service and other (total)','product_service',true,false,true),
 ('Microsoft','ProductOrService','Devices','Devices','product_service',false,false,true),
 ('Microsoft','ProductOrService','Dynamics','Dynamics','product_service',false,false,true),
 ('Microsoft','ProductOrService','DynamicsProductsAndCloudServices','Dynamics products and cloud services','product_service',false,false,true),
 ('Microsoft','ProductOrService','EnterpriseAndPartnerServices','Enterprise and partner services','product_service',false,false,true),
 ('Microsoft','ProductOrService','EnterpriseServices','Enterprise services','product_service',false,false,true),
 ('Microsoft','ProductOrService','Gaming','Gaming','product_service',false,false,true),
 ('Microsoft','ProductOrService','LinkedInCorporation','LinkedIn','product_service',false,false,true),
 ('Microsoft','ProductOrService','MicrosoftThreeSixFiveCommercialProductsAndCloudServices','Microsoft 365 Commercial products and cloud services','product_service',false,false,true),
 ('Microsoft','ProductOrService','MicrosoftThreeSixFiveConsumerProductsAndCloudServices','Microsoft 365 Consumer products and cloud services','product_service',false,false,true),
 ('Microsoft','ProductOrService','OfficeProductsAndCloudServices','Office products and cloud services','product_service',false,false,true),
 ('Microsoft','ProductOrService','OtherProductsAndServices','Other products and services','product_service',false,false,true),
 ('Microsoft','ProductOrService','SearchAndNewsAdvertising','Search and news advertising','product_service',false,false,true),
 ('Microsoft','ProductOrService','ServerProductsAndCloudServices','Server products and cloud services','product_service',false,false,true),
 ('Microsoft','ProductOrService','Windows','Windows','product_service',false,false,true),
 ('Microsoft','ProductOrService','WindowsAndDevices','Windows and Devices','product_service',false,false,true),
 ('Microsoft','ProductsOrServicesSecondaryCategorization','MicrosoftCloud','Microsoft Cloud (overlapping view)','excluded',false,false,false)
AS t(company, axis, member, segment_name, segment_type, in_top_level, in_detail_level, include);

In [0]:
-- 1) expect 56
SELECT COUNT(*) FROM silver.ref_segment_map;

-- 2) members in our facts that have NO mapping: expect no rows
SELECT DISTINCT f.company, f.axis, f.member
FROM silver.facts_revenue f
LEFT JOIN silver.ref_segment_map m
  ON m.company = f.company AND m.axis = f.axis AND m.member = f.member
WHERE f.dimn = 1 AND m.member IS NULL;

In [0]:
SHOW TABLES IN silver;

In [0]:
CREATE OR REPLACE TABLE silver.reconciliation AS
WITH seg_members AS (
  SELECT f.company, f.adsh, f.form, f.tag, f.period_end, f.qtrs, f.axis, f.value,
         'top' AS partition_name
  FROM silver.facts_revenue f
  JOIN silver.ref_segment_map m
    ON m.company = f.company AND m.axis = f.axis AND m.member = f.member
  WHERE f.dimn = 1 AND m.include AND m.in_top_level
  UNION ALL
  SELECT f.company, f.adsh, f.form, f.tag, f.period_end, f.qtrs, f.axis, f.value,
         'detail' AS partition_name
  FROM silver.facts_revenue f
  JOIN silver.ref_segment_map m
    ON m.company = f.company AND m.axis = f.axis AND m.member = f.member
  WHERE f.dimn = 1 AND m.include AND m.in_detail_level
),
seg AS (
  SELECT company, adsh, form, tag, period_end, qtrs, axis, partition_name,
         SUM(value) AS segment_sum, COUNT(*) AS members
  FROM seg_members
  GROUP BY company, adsh, form, tag, period_end, qtrs, axis, partition_name
),
tot AS (
  SELECT adsh, tag, period_end, qtrs, value AS reported_total
  FROM silver.facts_revenue
  WHERE dimn = 0
)
SELECT s.*, t.reported_total,
       s.segment_sum - t.reported_total AS diff,
       CASE WHEN t.reported_total IS NULL THEN 'NO_TOTAL'
            WHEN s.segment_sum = t.reported_total THEN 'PASS'
            ELSE 'FAIL' END AS status
FROM seg s
LEFT JOIN tot t
  ON t.adsh = s.adsh AND t.tag = s.tag
 AND t.period_end = s.period_end AND t.qtrs = s.qtrs;

In [0]:
SELECT company, axis, partition_name, status, COUNT(*) AS checks
FROM silver.reconciliation
GROUP BY company, axis, partition_name, status
ORDER BY company, axis, partition_name, status;

In [0]:
CREATE OR REPLACE TABLE silver.quarantine AS
SELECT f.*, 'value_not_numeric' AS reason
FROM silver.facts_revenue f
WHERE f.value IS NULL
UNION ALL
SELECT f.*, 'segment_text_malformed'
FROM silver.facts_revenue f
WHERE f.dimn = 1 AND (f.axis IS NULL OR f.axis = '' OR f.member IS NULL OR f.member = '')
UNION ALL
SELECT f.*, 'unmapped_member'
FROM silver.facts_revenue f
LEFT JOIN silver.ref_segment_map m
  ON m.company = f.company AND m.axis = f.axis AND m.member = f.member
WHERE f.dimn = 1 AND m.member IS NULL;

In [0]:
CREATE OR REPLACE TABLE silver.facts_revenue_latest AS
SELECT * EXCEPT (rn) FROM (
  SELECT
    f.*,
    fl.filed_date,
    m.segment_name,
    m.segment_type,
    ROW_NUMBER() OVER (
      PARTITION BY f.company, f.period_end, f.qtrs, f.dimn, f.axis, f.member
      ORDER BY fl.filed_date DESC,
               CASE f.tag
                 WHEN 'RevenueFromContractWithCustomerExcludingAssessedTax' THEN 1
                 WHEN 'Revenues' THEN 2
                 ELSE 3 END,
               f.adsh
    ) AS rn
  FROM silver.facts_revenue f
  JOIN silver.filings fl ON fl.adsh = f.adsh
  LEFT JOIN silver.ref_segment_map m
    ON m.company = f.company AND m.axis = f.axis AND m.member = f.member
  WHERE f.dimn = 0 OR m.include
)
WHERE rn = 1;

In [0]:
SELECT company, adsh, period_end, qtrs, dimn, axis, member,
       COUNT(DISTINCT tag) AS tags
FROM silver.facts_revenue
GROUP BY company, adsh, period_end, qtrs, dimn, axis, member
HAVING COUNT(DISTINCT tag) > 1;

In [0]:
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT concat_ws('|', company, period_end, qtrs, dimn, axis, member)) AS distinct_facts
FROM silver.facts_revenue_latest;

In [0]:
SELECT company, period_end, qtrs, value, form, filed_date
FROM silver.facts_revenue_latest
WHERE company = 'Amazon' AND dimn = 0 AND qtrs = 4
ORDER BY period_end;